In Python, a **decorator** is a design pattern that allows you to modify, extend, or supercharge the behavior of a function, method, or class **without permanently altering its actual source code.**

Syntactically, decorators are applied using the **`@`** symbol (often called "pie syntax") placed directly above a target function definition. Under the hood, they are simply higher-order functions that take a function as an input, wrap it with extra logic, and return a new, enhanced function.

---

- **🏎️ The Core Concept: Wrapper Logic**
    - To understand how a decorator works, think of a physical package. You have a core product (your function), and you place it inside a cardboard box with wrapping paper (the decorator). When someone interacts with the package, they hit the wrapping paper first before reaching the core product inside.

- **Step 1: The "Old School" Function Wrapper**
    - Because Python treats functions as first-class citizens, you can pass them around like any other object. Here is how a decorator works manually:

```python
def uppercase_decorator(func):
    # This nested function acts as the "wrapping paper"
    def wrapper():
        original_result = func()  # Execute the core function
        modified_result = original_result.upper()  # Inject custom logic
        return modified_result
    return wrapper  # Return the package

def greet():
    return "hello world"

# Manually wrapping the function
greet_extended = uppercase_decorator(greet)
print(greet_extended())  # Output: HELLO WORLD
```

- **Step 2: Clean Syntax with `@`**
    - Writing manual wrappers gets clunky quickly. Python introduces the `@` syntax to automatically pass the function through the decorator pipeline behind the scenes:

```python
@uppercase_decorator
def greet_clean():
    return "welcome home"

print(greet_clean())  # Output: WELCOME HOME
```

> By simply typing **`@uppercase_decorator`**, Python silently rewrites your code execution to match the manual wrapper pattern.

---

- **🚀 Handling Arguments with `*args` and `kwargs`**
    - The simple decorator above fails if your target function expects arguments. To make a decorator universal—capable of wrapping *any* function regardless of its signature—you must use argument unpacking inside the nested wrapper:

```python
def logger_decorator(func):
    def wrapper(*args, **kwargs):
        print(f"🎬 Calling function: '{func.__name__}'")
        result = func(*args, **kwargs)  # Captures any forwarded arguments
        print(f"✅ Finished execution.")
        return result
    return wrapper

@logger_decorator
def multiply(a, b):
    return a * b

print(multiply(4, 5))
# Output:
# 🎬 Calling function: 'multiply'
# ✅ Finished execution.
# 20
```

---

- **🛠️ Essential Real-World Utilities**
    - Decorators are incredibly powerful because they allow you to separate your core business logic from repetitive, application-wide infrastructure tasks.

- **1. Timing and Performance Benchmarking**
    - You can measure exactly how many milliseconds a process takes to execute without littering stopwatch timestamps inside every function manually.

```python
import time

def execution_timer(func):
    def wrapper(*args, **kwargs):
        start_time = time.perf_counter()
        result = func(*args, **kwargs)
        end_time = time.perf_counter()
        print(f"⏱️ '{func.__name__}' took {end_time - start_time:.4f} seconds to complete.")
        return result
    return wrapper
```

- **2. User Authentication and Permissions**
    - In web development (like Flask or Django), you can block unauthorized traffic from sensitive application endpoints using a gateway decorator.

```python
def require_login(func):
    def wrapper(user, *args, **kwargs):
        if not user.get("is_authenticated"):
            raise PermissionError("🔒 Access Denied: User is not logged in.")
        return func(user, *args, **kwargs)
    return wrapper

```

- **3. API Rate Limiting and Caching**
    - You can cache the outputs of heavy database queries or API requests so that if a user requests the exact same parameters a second time, the decorator bypasses computation entirely and serves the saved result instantly (`@functools.lru_cache` is a built-in version of this).

---

- **⚠️ Overcoming the Meta-Data Trap: `functools.wraps`**
    - When you wrap a function, its identity is hijacked by the inner `wrapper` function. If you inspect the metadata of a decorated function, you encounter an undesirable side effect:

```python
@logger_decorator
def say_hi():
    """This function greets the user."""
    return "Hi!"

print(say_hi.__name__)  # Output: wrapper (Instead of 'say_hi'!)
print(say_hi.__doc__)   # Output: None    (The docstring vanished!)
```

> This ruins debugging profiles and automated documentation engines. To fix this, Python provides **`@functools.wraps`**, a decorator designed specifically to copy the original function's name, docstring, and module traits back onto the wrapper:

```python
from functools import wraps

def polished_logger(func):
    @wraps(func)  # 🛡️ Preserves original function identity
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper
```

---

- **💎 Decorator Stacking (Chaining)**
    - You can apply multiple decorators to a single function. They execute in order from **top to bottom** (or inside-out mathematically):

```python
@uppercase_decorator
@logger_decorator
def get_status():
    return "online"
```

In this scenario, `get_status` runs through the performance logger first, and then that logged output stream is passed up into the uppercase transformer.

## **Decorating functions**

In [1]:
def decorator(func):
    return func

@decorator    # Basic Implementation of Decorators
def add(a, b):
    return a + b

In [2]:
add(1, 3)

4

In [21]:
import functools

In [22]:
def decorator(func):
    @functools.wraps(func)
    def _decorator(a, b):
        # Pass the modifier argument to the function
        result = func(a, b + 3)

        # Log the function call
        name = func.__name__
        print(f"{name}(a={a}, b={b}): {result}")

        # Return a modifier output
        return result + 3

    return _decorator

@decorator
def add(a, b):
    return a + b

In [23]:
add(1, 2)

add(a=1, b=2): 6


9

### **Generic function decorators**

In [24]:
def decorator(func):
    
    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        a, b = args

        return func(a, b + 3)

    return _decorator

@decorator
def add(a, b):
    return a + b

In [25]:
add(1, 2)

6

In [26]:
add(a=1, b=2)

ValueError: not enough values to unpack (expected 2, got 0)

> This code uses **positional-only arguments** (the / as the last function argument), which have been supported since Python 3.8. For older versions, you can emulate this behavior using *args instead of explicit arguments.

In [27]:
@decorator
def add(a, b, /):
    return a + b

In [28]:
add(1, 2)

6

In [29]:
add(a=1, b=2)

ValueError: not enough values to unpack (expected 2, got 0)

In [30]:
@decorator
def add(*, a, b):
    return a + b

In [31]:
add(1, 2)

TypeError: add() takes 0 positional arguments but 2 were given

In [32]:
import inspect

In [36]:
def decorator(func):
    # Use the inspect module to get function signature.
    sign = inspect.signature(func)

    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        # Bind the arguments to the given *args and **kwargs.
        # If you want to make arguments optional, use
        # signature.bind_partial instead.
        bound = sign.bind(*args, **kwargs)

        # Apply the defaults so b is always filled
        bound.apply_defaults()

        # Extract the filled arguments. If the number of
        # arguments is still expected to be fixed, you can use
        # tuple unpacking: 'a, b = bound.arguments.values()'
        a = bound.arguments['a']
        b = bound.arguments['b']
        return func(a, b + 3)

    return _decorator

@decorator
def add(a, b):
    return a + b

In [37]:
add(1, b=2)

6

In [39]:
add(a=1, b=2)

6

In [43]:
add(a=1, 2)

SyntaxError: positional argument follows keyword argument (1709895177.py, line 1)

### **The importance of functools.wraps**

In [47]:
# without functionamity of functools.wraps

def decorator(func):

    def _decorator(*args, **kwargs):
        return func(*args, **kwargs)

    return _decorator

@decorator
def add(a, b):
    """Add two number a and b"""

    return a + b

In [48]:
add(1, 2)

3

In [49]:
help(add)

Help on function _decorator in module __main__:

_decorator(*args, **kwargs)



In [50]:
add.__name__

'_decorator'

In [51]:
# Adding functools.wraps to the cods

def decorator(func):

    @functools.wraps(func)
    def _decorator(*args, **kwargs):
        return func(*args, **kwargs)

    return _decorator

@decorator
def add(a, b):
    """Add two number a and b"""

    return a + b

In [52]:
add(1, 2)

3

In [53]:
help(add)

Help on function add in module __main__:

add(a, b)
    Add two number a and b



In [54]:
add.__name__

'add'

The **`functools.wraps`** utility preserves a decorated function's identity by copying and updating its core metadata attributes to the underlying wrapper function.

- Instead of using complex magic, it performs a straightforward attribute transfer, explicitly copying:
    * **`__doc__`**: The function's docstring documentation.
    * **`__name__`**: The defined name of the function.
    * **`__module__`**: The module namespace where the function was declared.
    * **`__annotations__`**: The type hints applied to inputs and outputs.
    * **`__qualname__`**: The full qualified path name of the function.

> Additionally, it syncs custom function properties by updating the wrapper’s **`__dict__`** state with the original function's dictionary contents, and injects a new **`__wrapped__`** attribute that maintains a direct reference back to the original, unmodified function object.

### **Chaining or nesting decorators**

- When stacking multiple decorators onto a single function, the execution lifecycle functions like concentric layers of an onion, making the structural order critical to keep track of. During the compilation phase, the decorators are initialized and wrapped from the **inside out** (the decorator closest to the function definition executes first). However, when the final decorated function is actually called at runtime, the processing flow triggers from the **outside in** (the outermost wrapper executes its setup logic first before passing control down the chain). Finally, once the core function completes its run, the teardown phase reverses the pipeline, processing post-execution results from the **inside back out** to the outermost layer.

In [55]:
def track(func=None, label=None):
    if label and not func:
        return functools.partial(track, label=label)

    print(f"Initializing {label}")

    @functools.wraps(func)
    def _track(*args, **kwargs):
        print(f"Calling {label}")
        func(*args, **kwargs)
        print(f"called {label}")

    return _track

In [56]:
@track(label='outer')
@track(label='inner')
def func():
    print('function')

Initializing inner
Initializing outer


In [57]:
func()

Calling outer
Calling inner
function
called inner
called outer


- This code implements a flexible decorator capable of accepting optional keyword arguments by leveraging **`functools.partial`** to freeze the **`label`** argument and return a modified decorator blueprint when a function isn't initially supplied. When stacking multiple instances of this decorator, like **`@track(label='outer')`** and **`@track(label='inner')`**, they wrap the core function in concentric layers, resembling an onion structure.

- Consequently, during the initialization and preprocessing phase, execution flows from the **outermost wrapper down to the innermost wrapper** before the actual function triggers; once the core function completes, control bubbles back upward, executing the post-processing results phase in reverse order from the **innermost wrapper back up to the outermost wrapper**.

### **Registering functions using decorators**

In [58]:
from collections import defaultdict

In [59]:
class EventsRegistry:

    def __init__(self):
        self.registry = defaultdict(list)

    def on(self, *events):
        def _on(func):
            for event in events:
                self.registry[event].append(func)
            return func

        return _on

    def fire(self, event, *args, **kwargs):
        for func in self.registry[event]:
            func(*args, **kwargs)

In [60]:
events = EventsRegistry()

@events.on('success', 'error')
def teardown(value):
    print(f"Tearing down got: {value}")

@events.on('success')
def success(value):
    print(f'Successfully executed: {value}')

In [61]:
events.fire('non-existing', 'nothing to see here')

In [62]:
events.fire('error', 'Oops, some error here')

Tearing down got: Oops, some error here


In [63]:
events.fire('success', 'Everything is fine')

Tearing down got: Everything is fine
Successfully executed: Everything is fine


### **Memoization using decorators**

In [69]:
def memoize(func):
    # Store the cache as attribute of the function so we can
    # apply the decorator to multiple functions without
    # sharing the cache.
    func.cache = dict()

    @functools.wraps(func)
    def _memoize(*args):
        # If the cache is not available, call the function
        # Note that all args need to be hashable
        if args not in func.cache:
            func.cache[args] = func(*args)
        return func.cache[args]

    return _memoize

@memoize
def fibonacci(n):
    if n < 2:
        return n
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

In [70]:
for i in range(1, 7):
    print(f"fibonnaci {i}: {fibonacci(i)}")

fibonnaci 1: 1
fibonnaci 2: 1
fibonnaci 3: 2
fibonnaci 4: 3
fibonnaci 5: 5
fibonnaci 6: 8


In [71]:
fibonacci.__wrapped__.cache

{(1,): 1, (0,): 0, (2,): 1, (3,): 2, (4,): 3, (5,): 5, (6,): 8}

- While writing a custom memoization decorator is a valuable educational exercise, manually implementing it in production is generally redundant since Python 3.2 introduced the built-in **`functools.lru_cache`** (Least Recently Used cache). This native decorator is a more sophisticated framework utility that automatically caches a function's return values based on its arguments, allowing subsequent matching requests to bypass execution and return results instantly.

- Unlike a simple custom dictionary that grows indefinitely, **`lru_cache`** manages memory efficiently by maintaining a fixed size limit (defaulting to 128 entries) and evicting the oldest, least-requested data when full; furthermore, it tracks real-time performance statistics—such as hits and misses—giving developers the necessary insights to optimize and scale the cache size dynamically.

In [78]:
# Create a simple call counting decorator
def counter(func):
    func.calls = 0

    @functools.wraps(func)
    def _counter(*args, **kwargs):
        func.calls += 1
        return func(*args, **kwargs)

    return _counter

# Create a LRU cache with size 3
@functools.lru_cache(maxsize=3)
@counter
def fibonacci(n):
    if n < 2:
        return n
    else:
        return fibonacci(n - 1) + fibonacci(n - 2)

In [79]:
fibonacci(100)

354224848179261915075

In [81]:
fibonacci.cache_info()

CacheInfo(hits=98, misses=101, maxsize=3, currsize=3)

In [83]:
fibonacci.__wrapped__.__wrapped__.calls

101

### **Decorators with (optional) arguments**

In Python, a **callable** is simply any object that you can call using a pair of parentheses `()` and optionally pass arguments into.

If you can append `()` to an object and it executes some block of code without throwing a `TypeError: '...' object is not callable`, then that object is considered a callable.

---

- **🛠️ What Objects Are Callable?**
    - Many developers assume only standard functions are callable, but Python expands this definition to several types of objects:

- **1. Built-in and Custom Functions**
    - The most obvious callables are standard functions created with the `def` keyword or anonymous functions created via `lambda`.

```python
def greet():
    return "Hello!"

# Both are callables
greet()  
(lambda x: x * 2)(5)
```

- **2. Built-in and Custom Classes**
    - When you call a class, you are telling Python to instantiate and return a new object.

```python
# Calling the built-in 'list' class creates an empty list
my_list = list() 
```

- **3. Methods**
    - Methods are simply functions that are bound to an object instance (like `string.upper()` or `list.append()`).

- **4. Custom Objects (The `__call__` Magic Method)**
    - You can turn **any normal object instance** into a callable by defining the `__call__` magic method inside its class definition. This allows an object to retain state while behaving exactly like a function.

```python
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, number):
        return number * self.factor

# 1. Create the object instance
triple = Multiplier(3)

# 2. Call the object directly like a function!
print(triple(10))  # Output: 30
```

---

- **🔍 How to Check if an Object is Callable**
    - Python provides a built-in function named **`callable()`** that takes any object as an input and returns `True` if it can be called, and `False` if it cannot.

```python
print(callable(print))      # True (It's a built-in function)
print(callable(int))        # True (It's a class)
print(callable("hello"))    # False (Strings cannot be called)
```

In [88]:
def add(func=None, add_n=0):
    # function is not callable so it's probably 'add_n'
    if not callable(func):
        # Test to make sure we don't pass 'None' as 'add_n'
        if func is not None:
            add_n = func
        return functools.partial(add, add_n=add_n)

    @functools.wraps(func)
    def _add(n):
        return func(n) + add_n

    return _add

@add
def add_zero(n):
    return n

@add(add_n=1)
def add_one(n):
    return n

@add(add_n=2)
def add_two(n):
    return n

In [89]:
add_zero(8), add_one(8), add_two(8)

(8, 9, 10)

### **Creating decorators using classes**

In [90]:
class Debug(object):

    def __init__(self, func):
        self.func = func
        # functools.update_wrapper for classe
        functools.update_wrapper(self, func)

    def __call__(self, *args, **kwargs):
        output = self.func(*args, **kwargs)
        name = self.func.__name__
        print(f"{name}({args!r}, {kwargs!r}): {output!r}")
        return output

@Debug
def add(a, b=0):
    return a + b

In [91]:
add(5)

add((5,), {}): 5


5

In [92]:
add(4, 3)

add((4, 3), {}): 7


7

## **Decorating class functions**

In [93]:
def plus_one(func):
    @functools.wraps(func)
    def _plus_one(self, n, *args):
        return func(self, n + 1, *args)

    return _plus_one

class Adder(object):
    @plus_one
    def add(self, a, b=0):
        return a + b

In [94]:
adder = Adder()

adder.add(0)

1

In [95]:
adder.add(3, 4)

8

- When a decorator is applied to a method inside a class rather than a standalone function, the underlying execution mechanism remains completely unchanged, with the only distinction being how object instances are tracked. Because methods are technically functions bound to an object, the decorator's inner wrapper function automatically receives the instance reference—conventionally named **`self`**—as its absolute first positional argument.

- This means your decorator can inspect or modify the parent object's internal properties dynamically at runtime, ensuring that wrapping class methods integrates seamlessly with standard Object-Oriented patterns without requiring any specialized syntax modifications.

### **Skipping the instance – classmethod and staticmethod**

The fundamental difference between **`@classmethod`** and **`@staticmethod`** lies in **what information is automatically passed to the method** when it is called.

While both are decorators used to define methods inside a class that can be executed without instantiating an object first, they serve completely different structural purposes.

---

- **🏎️ The Quick Breakdown**
    * **`@classmethod`**: Receives the **class itself** (conventionally named `cls`) as its implicit first argument. It can access and modify class-level state.
    * **`@staticmethod`**: Receives **no implicit arguments** (neither `self` nor `cls`). It behaves exactly like a plain, isolated function that just happens to live inside the class's namespace.

---

- **🛠️ The Structural Difference in Code**

```python
class Demo:
    class_variable = "Shared Data"

    @classmethod
    def a_class_method(cls, arg1):
        # Has access to the class namespace via 'cls'
        return f"Class method called. Accessing: {cls.class_variable}"

    @staticmethod
    def a_static_method(arg1):
        # Completely isolated. Cannot access 'self' or 'cls'
        return f"Static method called with argument: {arg1}"
```

---

- **🚀 When to Use Which? (The True Intent)**

- **1. Use `@classmethod` for Factory Methods (Alternative Constructors)**
    - The absolute highest utility of a `@classmethod` is to create **alternative constructors** for your class. If your class normally accepts specific variables, but you sometimes want to instantiate it using data from a JSON payload, a CSV row, or a formatted string, you use a class method.

```python
class User:
    def __init__(self, first_name, last_name):
        self.first_name = first_name
        self.last_name = last_name

    @classmethod
    def from_string(cls, full_name_str):
        # Parses data first, then uses 'cls' to build and return a new instance
        first, last = full_name_str.split(" ")
        return cls(first, last)  # Identical to calling User(first, last)

# Standard creation
user1 = User("John", "Doe")

# Alternative creation via Class Method Factory
user2 = User.from_string("Jane Smith")
```

* *Why this rules:* If you ever inherit from this class (**`class PowerUser(User):`**), **`cls`** automatically updates to point to the subclass **`PowerUser`**, making your constructors perfectly reusable and dynamic.

- **2. Use `@staticmethod` for Isolated Helper Functions**
    - A `@staticmethod` is used when you need a utility function that is logically connected to the class topic, but doesn't actually need to read or change any properties belonging to the class or its objects.

```python
class DateValidator:
    @staticmethod
    def is_valid_date(date_str):
        # Simply evaluates an input string; doesn't care about class or object state
        if len(date_str) == 10 and "-" in date_str:
            return True
        return False

# You can use it instantly without instantiating the class
print(DateValidator.is_valid_date("2026-06-02"))  # True
```

---

- **⚖️ Summary Comparison Matrix**

| Feature | `@classmethod` | `@staticmethod` |
| --- | --- | --- |
| **Implicit First Argument** | Yes, the class object (`cls`) | No |
| **Access Class Variables?** | **Yes**, via `cls.variable_name` | No (unless you hardcode `ClassName.var`) |
| **Access Instance Properties?** | No | No |
| **Primary Use Case** | Alternative constructors / Factories | Namespace isolation for helper utilities |
| **Subclassing Behavior** | Adapts dynamically to the subclass | Remains bound to the hardcoded method logic |

In [96]:
from pprint import pprint

In [97]:
class Spam(object):
    def some_instancemethod(self, *args, **kwargs):
        pprint(locals(), width=60)

    @classmethod
    def some_classmethod(cls, *args, **kwargs):
        pprint(locals(), width=60)

    @staticmethod
    def some_staticmethod(*args, **kwargs):
        pprint(locals(), width=60)

In [98]:
spam = Spam()

In [99]:
spam.some_instancemethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'kwargs': {'a': 3, 'b': 4},
 'self': <__main__.Spam object at 0x7f98a8589400>}


In [101]:
Spam.some_instancemethod()

TypeError: Spam.some_instancemethod() missing 1 required positional argument: 'self'

In [102]:
Spam.some_instancemethod(1, 2, a=3,  b=4)

{'args': (2,), 'kwargs': {'a': 3, 'b': 4}, 'self': 1}


In [103]:
spam.some_classmethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'cls': <class '__main__.Spam'>,
 'kwargs': {'a': 3, 'b': 4}}


In [104]:
Spam.some_classmethod(1, 2, a=3,  b=4)

{'args': (1, 2),
 'cls': <class '__main__.Spam'>,
 'kwargs': {'a': 3, 'b': 4}}


In [105]:
spam.some_staticmethod()

{'args': (), 'kwargs': {}}


In [106]:
spam.some_staticmethod(1, 2, a=3,  b=4)

{'args': (1, 2), 'kwargs': {'a': 3, 'b': 4}}


In [107]:
Spam.some_staticmethod()

{'args': (), 'kwargs': {}}


In [108]:
Spam.some_staticmethod(1, 2, a=3,  b=4)

{'args': (1, 2), 'kwargs': {'a': 3, 'b': 4}}


- Before diving deeper into decorators, it is essential to understand **Python descriptors**, which are protocols that allow you to customize and hijack the default binding behavior of object attributes. Normally, accessing an attribute simply reads or writes its data directly from the object's internal dictionary; however, if a descriptor object is assigned as the value of an attribute instead, it intercepts those standard lookups.

- By defining specific magic methods—namely **`__get__()`**, **`__set__()`**, and **`__delete__()`** the descriptor acts as a gatekeeper, giving you complete programmatic control over precisely what happens, what values are calculated, and what validation checks occur whenever that attribute is accessed, modified, or removed from the instance.

In [112]:
class Spam:
    def __init__(self, spam):
        self.spam = spam

    def __get__(self, instance, cls):
        return self.spam + instance.eggs

    def __set__(self, instance, value):
        instance.eggs = value - self.spam

class Sandwich:
    spam = Spam(5)

    def __init__(self, eggs):
        self.eggs = eggs

In [113]:
sanwich = Sandwich(1)

In [114]:
sanwich.eggs

1

In [115]:
sanwich.spam

6

In [116]:
sanwich.eggs = 10
sanwich.spam

15

In [123]:
class ClassMethod(object):
    def __init__(self, method):
        self.method = method

    def __get__(self, instancen, cls):
        @functools.wraps(self.method)
        def method(*args, **kwargs):
            return self.method(cls, *args, **kwargs)

        return method


class StaticMethod(object):
    def __init__(self, method):
        self.method = method

    def __get__(self, instance, cls):
        return self.method

In [124]:
class Sandwich:
    spam = "Spam"

    def __init__(self, spam):
        self.spam = spam

    @ClassMethod
    def some_classmethod(cls, args):
        return cls.spam, args

    @StaticMethod
    def some_staticmethod(args):
        return Sandwich.spam, args

In [125]:
sandwich = Sandwich('instance')

sandwich.spam

'instance'

In [126]:
sandwich.some_classmethod('argument')

('Spam', 'argument')

In [127]:
sandwich.some_staticmethod('argument')

('Spam', 'argument')

### **Properties – Smart descriptor usage**

> Python 3.8 added **functools.cached_property**, which functions the same as property but executes only once per instance.

In [135]:
class Sandwich(object):

    def get_eggs(self):
        print("getting eggs")
        return self._eggs

    def set_eggs(self, eggs):
        print("setting eggs to %s" % eggs)
        self._eggs = eggs

    def deled_eggs(self):
        print("deleting eggs")
        del self._eggs

    eggs = property(get_eggs, set_eggs, deled_eggs)

    @property
    def spam(self):
        return self._spam

    @spam.setter
    def spam(self, spam):
        print("setting spam to %s" %spam)
        self._spam = spam

    @spam.deleter
    def spam(self):
        print("deleting spam")
        del self._spam

    @functools.cached_property
    def becon(self):
        print("getting becon")
        return 'becon!'

In [136]:
sandwich = Sandwich()

In [137]:
sandwich.eggs = 123

setting eggs to 123


In [138]:
sandwich.eggs

getting eggs


123

In [139]:
del sandwich.eggs

deleting eggs


In [140]:
sandwich.becon

getting becon


'becon!'

In [141]:
sandwich.becon

'becon!'

In [150]:
class Property(object):
    def __init__(self, fget=None, fset=None, fdel=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel

    def __get__(self, instance , cls):
        if isinstance is None:
            return self
        elif self.fget:
            return self.fget(instance)

    def __set__(self, instance, value):
        self.fset(instance, value)

    def __delete__(self, instance):
        self.fdel(insatnce)

    def getter(self, fget):
        return Property(fget, self.fset, self.fdel)

    def setter(self, fset):
        return Property(self.fget, fset, self.fdel)

    def deleter(self, fdel):
        return Property(self.fget, self.fset, fdel)

In [151]:
class Sandwich:
    @Property
    def eggs(self):
        return self._eggs

    @eggs.setter
    def eggs(self, value):
        self._eggs = value

    @eggs.deleter
    def eggs(self):
        del self._eggs

In [152]:
sandwich = Sandwich()

In [153]:
sandwich.eggs = 5

In [154]:
sandwich.eggs

5

In [161]:
class Sandwich(object):
    def __init__(self):
        self.registry = {}

    def __getattr__(self, key):
        print("getting %r" %key)
        return self.registry.get(key, "Undefined")

    def __setattr__(self, key, value):
        if key == 'registry':
            object.__setattr__(self, key, value)
        else:
            print("Setting %r to %r" %(key, value))
            self.registry[key] = value

    def __delattr__(self, key):
        print("Deletting %r" %key)
        del self.registry[key]

In [162]:
sandwich = Sandwich()

In [163]:
sandwich.a

getting 'a'


'Undefined'

In [164]:
sandwich.a = 1

Setting 'a' to 1


In [165]:
sandwich.a

getting 'a'


1

In [166]:
del sandwich.a

Deletting 'a'


- In Python object internals, the core difference between **`__getattr__`** and **`__getattribute__`** lies in their execution triggers, which introduces critical safety implications when managing attribute access. The `__getattr__` method acts as a fallback gatekeeper, executing *only* after Python searches the object's internal `__dict__` and fails to find the requested key, which is why it ignores existing attributes.

- In contrast, **`__getattribute__` intercepts *every single attribute lookup* indiscriminately, making it far more powerful but highly volatile; if you try to reference an internal attribute like `self.registry` within it, the call triggers `__getattribute__` again, creating an infinite recursive loop that will crash your program unless you implement explicit exclusion logic. While developers rarely need to construct these descriptor mechanisms manually, they serve as the vital architectural foundation for several core Python processes, such as resolving inheritance paths via the `super()` method.

## **Decorating classes**

- Introduced in **Python 2.6**, the class decorator syntax is a structural alternative to standard inheritance, mixins, or metaclasses that allows developers to dynamically modify or augment a class definition at declaration time rather than at instantiation time. Just like a standard function decorator, a class decorator is fundamentally a higher-order callable that takes a regular class object as its input, executes modifications or wraps its behavior, and returns a decorated class.

- While the syntax simplifies the design pattern, it is not a completely new technique under the hood, as it merely automates the manual assignment of passing a class directly into a wrapping function **(`DecoratedClass = decorator(RegularClass)`**). Because Python already provides robust, built-in object-oriented features to manipulate class structures—such as traditional subclassing—decorators are never strictly mandatory, which explains why they are less frequently encountered in production codebases despite their immense utility for clean code reuse.

A **mixin** is a software design pattern in object-oriented programming where a class provides specific, tightly focused functionality to be inherited by other classes, but is **not intended to stand on its own.** Think of a mixin as a discrete "plug-in" or a "feature module." You don't build an entire application out of mixins; instead, you inject them into primary classes to grant them specialized abilities without using heavy, rigid parent-child inheritance hierarchies.

---

- **🏎️ The Core Problem: The Inheritance Trap**
    - Imagine you are building a game with various entities: `Hero`, `Enemy`, `NPC`, and `TreasureChest`.
    - You want both the `Hero` and the `Enemy` to have a `kill()` method because they can die. However, an `NPC` cannot die, and a `TreasureChest` certainly can't.
    - If you use traditional hierarchical inheritance, you run into trouble:
        * If you put `kill()` in a master `GameObject` parent class, then `TreasureChest` accidentally inherits the ability to die.
        * If you create a `MortalGameObject` subclass, your inheritance tree starts splitting and becomes a tangled mess once you try adding other shared traits (like `CanInventory`, `CanSpells`, etc.).

---

- **💡 The Solution: Combining Mixins**
    - Instead of forcing classes into a strict *"is-a"* family tree, mixins allow you to build classes based on a *"has-a"* or *"can-do"* composition mindset. You write small, isolated classes that do exactly *one* thing well, and then multi-inherit them into your main classes.

- **The Code Implementation**
    - In Python, a mixin is just a normal class. By convention, developers usually append the word `Mixin` to its name to clarify its architectural role to the rest of the team:

```python
class KillableMixin:
    """Provides health tracking and death mechanics."""
    def __init__(self):
        self.health = 100

    def take_damage(self, amount):
        self.health -= amount
        if self.health <= 0:
            self.die()

    def die(self):
        print(f"💀 {self.__class__.__name__} has been destroyed!")


class InventoryMixin:
    """Provides container and item storage mechanics."""
    def __init__(self):
        self.items = []

    def pick_up(self, item):
        self.items.append(item)
        print(f"🎒 Added {item} to inventory.")
```

> Now, you can compose your actual game objects by mixing and matching these traits seamlessly using **multiple inheritance**:

```python
# The Hero needs BOTH combat mechanics and an inventory
class Hero(KillableMixin, InventoryMixin):
    def __init__(self, name):
        self.name = name
        # Initialize the mixins' internal states
        KillableMixin.__init__(self)
        InventoryMixin.__init__(self)

# A Treasure Chest only needs an inventory—it cannot die
class TreasureChest(InventoryMixin):
    def __init__(self):
        InventoryMixin.__init__(self)

# --- Gameplay Execution ---
player = Hero("Aragorn")
player.pick_up("Andúril Sword") # Granted by InventoryMixin
player.take_damage(100)         # Granted by KillableMixin

chest = TreasureChest()
chest.pick_up("100 Gold Coins")
# chest.take_damage(50) ❌ AttributeError: 'TreasureChest' object has no attribute 'take_damage'
```

---

- **⚖️ Rules for Writing Mixins**
    - To prevent your multiple inheritance structures from devolving into unmaintainable spaghetti code, follow these strict guidelines:
        1. **Never Instantiate a Mixin Directly:** A mixin is an incomplete fragment. Writing `my_mixin = KillableMixin()` should be useless on its own.
        2. **No Monolithic State:** Mixins should generally avoid holding massive independent state variables unless absolutely required for their explicit feature (like the `self.items` array above).
        3. **Beware of the MRO (Method Resolution Order):** Python resolves inherited methods from left to right. If two mixins accidently share a method with the exact same name, the leftmost mixin in your class definition wins. Keeping mixins hyper-focused ensures their method names never clash.

### **Singletons – Classes with a single instance**

In [1]:
import functools

In [2]:
def singletons(cls):
    instance = dict()
    
    @functools.wraps(cls)
    def _singletons(*args, **kwargs):
        if cls not in instance:
            instance[cls] = cls(*args, **kwargs)
        return instance[cls]

    return _singletons

In [3]:
@singletons
class SomeSingletons(object):
    def __init__(self):
        print("executing init")

In [4]:
a = SomeSingletons()

executing init


In [5]:
b = SomeSingletons()

In [6]:
a is b

True

In [8]:
a.x = 123

b.x

123

> The is operator compares objects by identity, which **is** implemented as the memory address in **CPython**. If a is b returns True, we can conclude that both a and b are the same instance.

### **Total ordering – Making classes sortable**

In [9]:
class Value(object):
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f"<{self.__class__.__name__} {self.value}>"

class Spam(Value):
    def __gt__(self, other):
        return self.value > other.value

    def __lt__(self, other):
        return self.value < other.value

    def __ge__(self, other):
        return self.value >= other.value

    def __le__(self, other):
        return self.value <= other.value

    def __eq__(self, other):
        return self.value == other.value

@functools.total_ordering
class Egg(Value):
    def __lt__(self, other):
        return self.value < other.value

    def __eq__(self, other):
        return self.value == other.value

In [10]:
numbers = [8, 2, 3, 8]
spams = [Spam(n) for n in numbers]
eggs = [Egg(n) for n in numbers]

In [11]:
spams

[<Spam 8>, <Spam 2>, <Spam 3>, <Spam 8>]

In [12]:
eggs

[<Egg 8>, <Egg 2>, <Egg 3>, <Egg 8>]

In [13]:
sorted(spams)

[<Spam 2>, <Spam 3>, <Spam 8>, <Spam 8>]

In [14]:
sorted(eggs)

[<Egg 2>, <Egg 3>, <Egg 8>, <Egg 8>]

In [15]:
values = [Value(n) for n in numbers]

values

[<Value 8>, <Value 2>, <Value 3>, <Value 8>]

In [17]:
sorted(values, key=lambda v: v.value)

[<Value 2>, <Value 3>, <Value 8>, <Value 8>]

In [42]:
def sort_by_attribute(attr, keyfunc=getattr):
    def _sort_by_attribute(cls):
        def __lt__(self, other):
            return getattr(self, attr) < getattr(other, attr)

        def __eq__(self, other):
            return getattr(self, attr) == getattr(other, attr)

        cls.__lt__ = __lt__
        cls.__eq__ = __eq__

        return functools.total_ordering(cls)

    return _sort_by_attribute

In [43]:
class Value(object):
    def __init__(self, value):
        self.value = value

    def __repr__(self):
        return f"<{self.__class__.__name__} {self.value}>"

In [44]:
@sort_by_attribute('value')
class Spam(Value):
    pass

In [45]:
numbers = [8, 4, 3, 8]
spams = [Spam(n) for n in numbers]

spams

[<Spam 8>, <Spam 4>, <Spam 3>, <Spam 8>]

In [46]:
sorted(spams)

[<Spam 3>, <Spam 4>, <Spam 8>, <Spam 8>]

## **Useful decorators**

Python includes several highly optimized, built-in decorators within its standard library. These tools solve common architectural challenges—such as memory management, optimization, object-oriented structuring, and type safety—without requiring you to write custom wrapper logic.

---

- **1. `functools.lru_cache` (The Performance Booster)**
    - The **Least Recently Used (LRU) Cache** is one of the most powerful decorators for optimization. It implements **memoization**, meaning it remembers the inputs and outputs of a function. If you call the function again with the exact same arguments, it bypasses the function execution entirely and serves the result instantly from memory.

- **Ideal Use Case:**
    - Expensive computations, recursive algorithms (like Fibonacci or grid travel), or repetitive, static database/API queries.

```python
from functools import lru_cache
import time

# Set a maxsize limit to prevent infinite memory growth
@lru_cache(maxsize=128)
def heavy_calculation(n):
    time.sleep(2)  # Simulating a slow network or CPU load
    return n * 42

# First run: Takes 2 seconds
print(heavy_calculation(10)) 

# Second run: Takes 0.0000 seconds (served instantly from cache!)
print(heavy_calculation(10)) 
```

---

- **2. `functools.cached_property` (The On-Demand Computer)**
    - Introduced in **Python 3.8**, **`@cached_property`** transforms a class method into a property that is computed **exactly once** per object instance. The first time you look up the property, it runs the code and caches the result directly on the object as a normal attribute. Subsequent lookups simply read the attribute without re-running the method.

- **Ideal Use Case:**
    - Object properties that are expensive to compute (e.g., parsing a large file or computing data metrics) but are guaranteed never to change once calculated.

```python
from functools import cached_property

class Dataset:
    def __init__(self, raw_data):
        self.raw_data = raw_data

    @cached_property
    def clean_metrics(self):
        print("🧼 Running heavy data cleaning metrics...")
        return [x for x in self.raw_data if x > 0]

data = Dataset([-10, 5, 20, -3])

# First access: Prints the message and runs the loop
print(data.clean_metrics)  # Output: [5, 20]

# Second access: Runs instantly, no loop executed
print(data.clean_metrics)  # Output: [5, 20]
```

---

- **3. `dataclasses.dataclass` (The Boilerplate Eradicator)**
    - Writing classes that primarily store data can result in massive amounts of boilerplate code (manually defining **`__init__`**, **`__repr__`**, **`__eq__`**, etc.). The **`@dataclass`** decorator automatically generates all of these fundamental structural methods under the hood based on your class-level type hints.

- **Ideal Use Case:**
    - Creating database models, API payload structures, or configurations cleanly.

```python
from dataclasses import dataclass

@dataclass
class Product:
    name: str
    price: float
    stock: int

# Generates __init__ automatically
item = Product("Mechanical Keyboard", 120.00, 5)

# Generates a beautiful __repr__ automatically
print(item)  # Output: Product(name='Mechanical Keyboard', price=120.0, stock=5)
```

---

- **4. `contextlib.contextmanager` (The Resource Handler)**
    - This decorator allows you to build clean, custom context managers using a simple generator function (`yield`) instead of creating a heavy class structured around `__enter__` and `__exit__` magic methods.

- **Ideal Use Case:**
    - Safely locking resources, handling database connections, or altering a state configuration environment temporarily before rolling it back.

```python
from contextlib import contextmanager

@contextmanager
def database_transaction(db_connection):
    print("🚀 Opening SQL Transaction...")
    try:
        yield db_connection  # Hands control back to the 'with' block
        print("💾 Committing Changes safely.")
    except Exception as e:
        print(f"❌ Rolling back changes due to error: {e}")
        raise

# Usage:
# with database_transaction(conn) as tx:
#     tx.execute("INSERT INTO users...")
```

---

- **5. `property` (The Object Gatekeeper)**
    - The built-in `@property` decorator allows you to define a method but access it precisely like a normal public attribute. This allows you to add data validation, transformation, or read-only access controls to attributes seamlessly without breaking your public API interface.

- **Ideal Use Case:**
    - Enforcing business logic validation or computing derived fields on object variables dynamically.

```python
class Account:
    def __init__(self, balance):
        self._balance = balance

    @property
    def balance(self):
        """Getter: Access balance like an attribute."""
        return f"${self._balance:,.2f}"

    @balance.setter
    def balance(self, value):
        """Setter: Intercepts adjustments to add strict safety validation checks."""
        if value < 0:
            raise ValueError("❌ Account balance cannot be negative!")
        self._balance = value

acc = Account(1000)
print(acc.balance)  # Output: $1,000.00

# Triggers the setter method validation behind the scenes
acc.balance = 2500 
# acc.balance = -50  ❌ Throws ValueError
```

### **Single dispatch – Polymorphism in Python**

In Python, **`functools.singledispatch`** is a decorator that allows you to implement **function overloading** based on the data type of the *first argument*.

In many object-oriented languages (like Java or C++), you can write multiple functions with the exact same name but different argument types, and the compiler automatically selects the correct one. Python doesn't natively support this because it evaluates arguments dynamically. Without `singledispatch`, handling different data types usually forces you into writing complex, unreadable `if/elif/else` type-checking blocks.

---

- **🏎️ The Problem: The `isinstance` Anti-Pattern**
    - Imagine you are building a data processing engine that formats data differently depending on whether it receives a string, an integer, a list, or a dictionary. The standard approach quickly turns into a messy pyramid of type checks:

```python
# ❌ The Hard-to-Maintain Way
def process_data(data):
    if isinstance(data, str):
        return f"String: {data.strip()}"
    elif isinstance(data, int):
        return f"Integer processed: {data * 10}"
    elif isinstance(data, list):
        return [process_data(item) for item in data]
    else:
        return f"Unknown type: {repr(data)}"
```

> This is rigid, breaks the Open-Closed Principle (you have to modify the core function every time you support a new data type), and scales poorly.

---

- **💡 The Solution: `singledispatch` Clean Architecture**
    - With `@singledispatch`, you define a clean **fallback function** for unsupported types, and then register dedicated, isolated worker functions for each explicit data type.

```python
from functools import singledispatch

@singledispatch
def process_data(data):
    """The base/fallback implementation if no type matches."""
    return f"Unknown type: {repr(data)}"

@process_data.register(str)
def _(data):
    """Handles strings."""
    return f"String: {data.strip()}"

@process_data.register(int)
def _(data):
    """Handles integers."""
    return f"Integer processed: {data * 10}"

@process_data.register(list)
def _(data):
    """Handles lists recursively."""
    return [process_data(item) for item in data]
```

- **🚀 Executing the Code:**
    - You call the exact same function name (`process_data`), and Python automatically routes the execution to the correct type handler behind the scenes:

```python
print(process_data("  Hello  "))  # Output: String: Hello
print(process_data(5))            # Output: Integer processed: 50
print(process_data([1, "Hi"]))    # Output: [Integer processed: 10, 'String: Hi']
print(process_data(3.14))         # Output: Unknown type: 3.14 (Falls back to base)

```

---

- **🛠️ Advanced Usage & Modern Python Type Hints**
    - If you are using modern Python with type annotations, you don't even need to pass the type directly into the `.register()` decorator. If you omit it, `singledispatch` will automatically infer the target type from your argument's type hint:

```python
@process_data.register
def _(data: dict):
    """Infers 'dict' automatically from the type hint annotation."""
    return f"Dictionary Keys: {list(data.keys())}"
```

---

- **⚠️ Key Limitations to Keep in Mind**
    1. **Single Argument Dispatch Only:** It *only* looks at the data type of the **first positional argument** (`args[0]`). If you need to overload functions based on a combination of multiple argument types (e.g., doing one thing if `arg1` is a string AND `arg2` is an integer), you need full *multiple dispatch*, which requires external libraries like `multipledispatch`.
    2. **Object-Oriented Variance (`singledispatchmethod`):** If you try to use `@singledispatch` on a method inside a class, it will break. This is because a class method's first argument is always `self` (the object instance), so it will constantly try to dispatch based on the class type rather than your data. To fix this, Python provides a dedicated twin decorator: **`from functools import singledispatchmethod`**.

In [54]:
@functools.singledispatch
def show_type(argument):
    print(f"argument: {argument}")

@show_type.register(int)
def show_int(argument):
    print(f"int argument: {argument}")

@show_type.register
def show_float(argument: float):
    print(f"flaot argument: {argument}")

In [55]:
show_type("abc")

argument: abc


In [57]:
show_type(123)

int argument: 123


In [58]:
show_type(1.23)

flaot argument: 1.23


In [62]:
registry = dict()

def register(func):
    # Fetch the first type from the type annotation but be
    # careful not to overwrite the 'type' function
    type_ = next(iter(func.__annotations__.values()))
    registry[type_] = func

    @functools.wraps(func)
    def _register(argument):
        # Fetch the function using the type of argument, and
        # fall back to the main function
        new_func = registry.get(type(argument), func)
        return new_func(argument)

    return _register

@register
def show_type(argument: any):
    print(f"argument: {argument}")

@register
def show_int(argument: int):
    print(f"int argument: {argument}")

In [61]:
show_type("abc")

argument: abc


In [63]:
show_type(123)

int argument: 123


> When naming the functions, make sure that you do not overwrite the original **singledispatch** function. If you named **show_int** as just **show_type**, it would **overwrite** the initial **show_type** function. This would make it impossible to access the original **show_type** function and make all register operations after that fail as well.

In [64]:
import json

In [66]:
@functools.singledispatch
def write_as_json(file, data):
    json.dump(data, file)

@write_as_json.register(str)
@write_as_json.register(bytes)
def write_as_json_filename(file, data):
    with open(file, 'w') as fh:
        write_as_json(fh, data)

In [67]:
data = dict(a=1, b=2, c=3)

data

{'a': 1, 'b': 2, 'c': 3}

In [68]:
write_as_json('file1.json', data)

In [69]:
write_as_json(b'file2.json', data)

In [71]:
with open('file3.json', 'w') as fh:
    write_as_json(fh, data)

In [74]:
write_as_json.registry

mappingproxy({object: <function __main__.write_as_json(file, data)>,
              bytes: <function __main__.write_as_json_filename(file, data)>,
              str: <function __main__.write_as_json_filename(file, data)>})

In [75]:
write_as_json.registry.keys()

dict_keys([<class 'object'>, <class 'bytes'>, <class 'str'>])

### **contextmanager — with statements made easy**

In Python, a **context manager** is a resource management tool designed to ensure that system resources are properly allocated and automatically cleaned up exactly when needed.

The most common way to interact with a context manager is using the **`with`** statement. This architecture pattern handles the setup and teardown phases of resource management, ensuring that files are closed, network sockets are released, and database connections are terminated—**even if your code throws an unexpected runtime error.**

---

- **❌ The Problem: Manual Clean Up is Dangerous**
    - When dealing with external system resources, you must explicitly close them to prevent memory leaks or file locking issues. A naive implementation relies on manual management:

```python
# ❌ Risky: If an error happens on line 3, the file never closes!
file = open("data.txt", "w")
file.write("Processing sensitive system logs...")
# If an exception occurs here, the line below is skipped entirely
file.close()
```

> The old-school procedural solution to fix this required verbose, clunky `try/finally` block boilerplate logic:

```python
file = open("data.txt", "w")
try:
    file.write("Processing sensitive system logs...")
finally:
    file.close()  # Guaranteed to execute, but ugly and repetitive
```

---

- **💡 The Solution: The Modern `with` Statement**
    - The `with` statement completely abstracts away the `try/finally` infrastructure, making resource cleanup implicit, clean, and declarative:

```python
with open("data.txt", "w") as file:
    file.write("Processing sensitive system logs...")
# The file automatically slams closed the millisecond we step out of this indentation block
```

---

- **🛠️ Way 1: Creating a Context Manager with `@contextmanager`**
    - You don't have to limit yourself to built-in system tools like `open()`. Python's **`contextlib`** module allows you to turn any standard generator function into a fully operational custom context manager using a single decorator and a `yield` statement.

```python
from contextlib import contextmanager

@contextmanager
def manage_database_session(db_name):
    # --- STEP 1: SETUP PHASE ---
    print(f"🔌 Connecting to database: {db_name}...")
    connection = f"ActiveConn[{db_name}]"
    
    try:
        # --- STEP 2: HAND OFF CONTROL ---
        yield connection  # The object yielded here becomes the target variable in 'as target'
        
    finally:
        # --- STEP 3: TEARDOWN PHASE ---
        print(f"🔒 Disconnecting from database safely...")

# --- Executing the Custom Context Manager ---
with manage_database_session("Production_DB") as session:
    print(f"🚀 Running operations using: {session}")

# Output:
# 🔌 Connecting to database: Production_DB...
# 🚀 Running operations using: ActiveConn[Production_DB]
# 🔒 Disconnecting from database safely...
```

---

- **🏗️ Way 2: The Class Protocol (`__enter__` and `__exit__`)**
    - If your context manager needs complex state management, deep class properties, or intricate error-handling routines, you can implement the formal **Context Manager Protocol** on a custom class by defining two dunder methods:
        - 1. **`__enter__(self)`**: Executes the setup code and returns the object you want to use inside the `with` statement block.
        - 2. **`__exit__(self, exc_type, exc_val, exc_tb)`**: Executes the teardown code. It receives three arguments that details any exception thrown inside the execution block. If it returns `True`, it suppresses the exception; if it returns `False` (or `None`), the exception bubbles up normally.

```python
class ManagedResource:
    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"🛠️ Allocating resource: {self.name}")
        return self  # Bound directly to the variable after 'as'

    def __exit__(self, exc_type, exc_val, exc_tb):
        print(f"🧹 Releasing resource: {self.name}")
        if exc_type:
            print(f"⚠️ An error occurred inside the block: {exc_val}")
        return True  # 🛡️ Returning True swallows the error, preventing code crashes

# --- Executing the Class Protocol ---
with ManagedResource("File_Scanner") as resource:
    print("⚡ Doing high-performance tasks...")
    raise ValueError("System network drop detected!") # Simulating a crash

print("✨ Application keeps running safely because __exit__ suppressed the error!")
```

---

- **🚀 Common Real-World Use Cases**
    * **Thread Locks:** Safely acquiring and releasing concurrent execution mutexes (`with threading.Lock():`).
    * **Environment Mocking:** Temporarily altering environment paths or system variables during automated unit test routines.
    * **Timing Operations:** Designing a custom execution benchmarker that tracks execution time from `__enter__` to `__exit__`.
    * **Temporary Directories:** Creating ephemeral scratch folders that automatically delete themselves once data processing wraps up (`tempfile.TemporaryDirectory()`).

In [1]:
class Open:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode

    def __enter__(self):
        self.handle = open(self.filename, self.mode)
        return self.handle

    def __exit__(self, exc_type, exc_val, exc_tab):
        self.handle.close()

In [4]:
with Open('test.txt', 'w') as fh:
    print("task Completed!", file=fh)

In [5]:
import contextlib

In [6]:
@contextlib.contextmanager
def open_context_manager(file_name, mode='r'):
    fh = open(file_name, mode)
    yield fh
    fh.close()

In [7]:
with open_context_manager('test1.txt', 'w') as fh:
    print("task Completed!", file=fh)

> With **file objects**, **database connections**, and **connections**, it is important to always have a **close()** call to clean up resources. In the case of a **file**, it tells the operating system to write the data to disk (as opposed to temporary buffers), and in the case of **network connections** and **database connections**, it releases the network connection and related resources on both ends. With database connections, it will also notify the server that the connection is no longer needed so that part is also handled gracefully.

> Without these calls, you can quickly run into **“too many open files”** or **“too many connections”** errors.

In [8]:
with contextlib.closing(open('test.txt', 'a')) as fh:
    print("another Task Completed!", file=fh)

In [9]:
@contextlib.contextmanager
def debug(name):
    print(f"Debugging... {name}")
    yield
    print(f"Finishing Debugging... {name}")

In [10]:
@debug("test")
def test():
    print("this is inside of our test function")

In [11]:
test()

Debugging... test
this is inside of our test function
Finishing Debugging... test


### **Validation, type checks, and conversions**

In [12]:
import inspect
import functools

In [13]:
def enforce_type_hints(func):
    # Construct the signature from the function which contains
    # the type annotations
    signature = inspect.signature(func)

    @functools.wraps(func)
    def _enforce_type_hints(*args, **kwargs):
        # Bind the arguments and apply the default values
        bound = signature.bind(*args, **kwargs)
        bound.apply_defaults()

        for key, value in bound.arguments.items():
            print(f"key: {key} -> value: {value}")
            params = signature.parameters[key]
            # The annotation should be a callable
            # type/function so we can cast as validation
            if params.annotation:
                bound.arguments[key] = params.annotation(value)
        return func(*bound.args, **bound.kwargs)

    return _enforce_type_hints

In [14]:
@enforce_type_hints
def sandwich(bacon: str, eggs: int):
    print(f"bacon: {bacon}, eggs: {eggs} ")

In [15]:
sandwich("milk", 3)

key: bacon -> value: milk
key: eggs -> value: 3
bacon: milk, eggs: 3 


In [18]:
sandwich("milk", "3.0")

key: bacon -> value: milk
key: eggs -> value: 3.0


ValueError: invalid literal for int() with base 10: '3.0'

### **Useless warnings – How to ignore them safely**

In [19]:
import warnings

In [20]:
def ignore_warnings(warning, count=None):
    
    def _ignore_warnings(func):
        
        @functools.wraps(func)
        def __ignore_warnings(*args, **kwargs):
            
            # Execute the code while catching all warnings
            with warnings.catch_warnings(record=True) as wr:
                # Catch all warnings of the given type
                warnings.simplefilter('always', warning)
                # Execute the function
                result = func(*args, **kwargs)

            # Re-warn all warnings beyond the expected count
            if count is not None:
                for w in wr[count:]:
                    warnings.warn(w.message)

            return result

        return __ignore_warnings

    return _ignore_warnings

In [21]:
@ignore_warnings(DeprecationWarning, count=1)
def spam():
    warnings.warn('deprecation 1', DeprecationWarning)
    warnings.warn('deprecation 2', DeprecationWarning)

In [22]:
# Note, we use catch_warnings here because doctests normally
# capture the warnings quietly
with warnings.catch_warnings(record=True) as ws:
    spam()
    
    for i, w in enumerate(ws):
        print(w.message)

deprecation 2


## **Summary**

This chapter demonstrates how decorators simplify codebase architectures by injecting complex, reusable behavior into otherwise straightforward functions. While writing a decorator introduces more upfront complexity than manually hardcoding features directly inside a single function, its true value lies in **scalability and modularity**—allowing you to apply the exact same behavioral logic across dozens of functions and classes seamlessly.

- The key architectural insights and core utilities of this chapter include:
    * **Smarter Code Utilities:** Decorators serve as highly versatile tools across modern development for system-wide tasks like **debugging**, automating input **validation**, providing **argument convenience** (pre-filling or translating inputs), and managing **output convenience** (formatting or casting returned data).
    * **The Absolute Rule of `functools.wraps`:** When wrapping any function, you must never forget to apply **`@functools.wraps`**.
    * **Mitigating Debugging Nightmares:** Because decorators modify runtime behaviors, troubleshooting them can already be inherently challenging; losing critical metadata attributes (like **`__name__`** or **`__doc__`**) makes diagnostic debugging significantly more difficult if left unprotected.